In [2]:
## Python Packages
!pip install langchain
!pip install langchain-ollama
!pip install langchain-chroma
!pip install chromadb
!pip install pypdf
!pip install pymupdf
!pip install pillow
!pip install pytesseract
!pip install pdf2image
!pip install sentence-transformers
!pip install unstructured


In [3]:
import os
import fitz
import pytesseract

from PIL import Image

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

from langchain_ollama import (
    OllamaEmbeddings,
    ChatOllama
)

In [4]:
# ==========================================
# CONFIGURATION
# ==========================================

import os

# Folder containing RBI PDFs, bank statements, screenshots
INPUT_DIR = r"D:\VISHNU\RAG_INPUTS"

# Chroma database location
VECTOR_DB_DIR = r"D:\VISHNU\RBI_VECTOR_DB"

# OCR executable
pytesseract.pytesseract.tesseract_cmd = r"D:\VISHNU\tesseract.exe"

# Ollama models
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2"

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

print("Input Folder :", INPUT_DIR)
print("Vector DB :", VECTOR_DB_DIR)

Input Folder : D:\VISHNU\RAG_INPUTS
Vector DB : D:\VISHNU\RBI_VECTOR_DB


In [5]:
# ==========================================
# OCR FUNCTIONS
# ==========================================

def extract_text_from_pdf(pdf_path):

    documents = []

    pages = convert_from_path(pdf_path, dpi=200)

    print(f"\nProcessing PDF: {os.path.basename(pdf_path)}")

    for idx, page in enumerate(pages):

        text = pytesseract.image_to_string(page)

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": pdf_path,
                    "page": idx + 1
                }
            )
        )

        print(f"Page {idx+1}/{len(pages)} completed")

    return documents


def extract_text_from_image(image_path):

    image = Image.open(image_path)

    text = pytesseract.image_to_string(image)

    return [
        Document(
            page_content=text,
            metadata={
                "source": image_path
            }
        )
    ]

In [6]:
# ==========================================
# INGEST DOCUMENTS
# ==========================================

def ingest_documents():

    docs = []

    pdfs = glob.glob(os.path.join(INPUT_DIR, "*.pdf"))

    pngs = glob.glob(os.path.join(INPUT_DIR, "*.png"))

    jpgs = glob.glob(os.path.join(INPUT_DIR, "*.jpg"))

    jpegs = glob.glob(os.path.join(INPUT_DIR, "*.jpeg"))

    all_files = pdfs + pngs + jpgs + jpegs

    if len(all_files) == 0:
        print("No files found.")
        return

    for file in all_files:

        try:

            if file.lower().endswith(".pdf"):
                docs.extend(extract_text_from_pdf(file))

            else:
                docs.extend(extract_text_from_image(file))

        except Exception as e:
            print(file, e)

    print(f"\nDocuments extracted: {len(docs)}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )

    chunks = splitter.split_documents(docs)

    print("Chunks created:", len(chunks))

    embeddings = OllamaEmbeddings(
        model=EMBED_MODEL
    )

    print("\nCreating Chroma DB...")

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=VECTOR_DB_DIR
    )

    print("\nVector Database Ready")

    return vectorstore

In [7]:
# ==========================================
# LOAD VECTOR DB
# ==========================================

def load_vector_db():

    embeddings = OllamaEmbeddings(
        model=EMBED_MODEL
    )

    db = Chroma(
        persist_directory=VECTOR_DB_DIR,
        embedding_function=embeddings
    )

    return db

In [8]:
# ==========================================
# RETRIEVER
# ==========================================

def get_retriever():

    db = load_vector_db()

    retriever = db.as_retriever(
        search_kwargs={"k":5}
    )

    return retriever

In [9]:
# ==========================================
# LLM
# ==========================================

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0
)

In [10]:
# ==========================================
# ASK QUESTION
# ==========================================

def ask_question(question):

    retriever = get_retriever()

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [d.page_content for d in docs]
    )

    prompt = f"""
You are an RBI banking assistant.

Answer ONLY from the provided context.

If the answer is not present in the context say:

"I could not find that information in the uploaded documents."

CONTEXT:
{context}

QUESTION:
{question}
"""

    response = llm.invoke(prompt)

    print("\n")
    print("="*70)
    print(response.content)
    print("="*70)

    return response.content

In [ ]:
# ==========================================
# INTERACTIVE CHAT
# ==========================================

while True:

    query = input("\nAsk Question : ")

    if query.lower() in [
        "stop",
        "exit",
        "quit"
    ]:
        print("Session Closed")
        break

    ask_question(query)


Ask Question :  what to do when my atm card is lost




When your ATM card is lost, please contact our 24/7 Customer Service number at +91-40-2700-0700 (or) 1800-102-6262. They will guide you on the next steps and provide a replacement card if needed. You can also visit any of our RBI Branches for assistance.



Ask Question :  What types of cards can be used at an ATM




At RBI, we accept a variety of cards for ATM transactions. These include:

1. Debit Cards
2. Credit Cards
3. Prepaid Cards



Ask Question :   what to do when my atm card is lost




When your ATM card is lost, please contact our 24/7 Customer Service number immediately. They will guide you through the process of reporting the loss and issuing a replacement card. You can also visit any RBI branch near you with a valid government-issued ID to obtain a new card.



Ask Question :  what is the customer service number to contact when ATM card is lost




The customer service number to contact when an ATM card is lost can be found on the back of the ATM card.



Ask Question :  I lost my ATM card what is the custer service number to contact




According to our bank's policy, if you lose your ATM card, you can contact our 24/7 Customer Service Number at +91-124-2364561 for assistance.
